# RQ2 Efficiency

How efficient is the directed search in reaching boundary inputs, in model evaluations and in the size of the edit? This notebook reproduces the crossing curve over the budget, the cost-of-crossing table, the paired cell-cluster differences and the wall-clock time per run.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from mmb.analysis import stats as st, paired as pr, export as ex

RESULTS = Path("../results") if Path("../results").exists() else Path("results")
FIG = RESULTS / "figures"; FIG.mkdir(parents=True, exist_ok=True)
GRID = ["qwen", "internvl", "ministral"]      # the three SUTs of the study (balanced grid)
REF = "a0_directed"                            # MMBA, the reference of every paired test
N_BOOT = 2000                                  # the paper uses 10000; raise for the final numbers
df = st.load_results([RESULTS / "MAIN.csv"])
scopes = ex.scopes(df, GRID)                   # {(sut, task): frame}; ("All", "All") pools the grid
grid_df = scopes[("All", "All")]
print(len(df), "runs;", "arms:", sorted(df["arm"].unique()))
from mmb.analysis import plots

## Crossing curve over the budget

Share of runs that crossed within b evaluations, one line per arm, with bootstrap bands. The jump at the left edge is the initial population.

In [ ]:
plots.plot_crossing_curve(grid_df, FIG / "crossing-curve.pdf", ref_arm=REF if REF in set(grid_df["arm"]) else None, vlines=[50], n_boot=N_BOOT)
st.curve_with_bands(grid_df, st.DEFAULT_BUDGETS, N_BOOT).pivot(index="arm", columns="budget", values="share").round(3)

## Cost of the crossing input

Fewest active genes among the crossing evaluations of a run (censored at the genotype length), the edit hypervolume, and the smallest image and text magnitudes. Magnitudes are only compared within one source (logged diagnostics), never mixed with fitness values.

In [ ]:
cost = ex.cost_summary(grid_df, N_BOOT)
cost

## Paired cost differences

Difference against the directed arm on matched pairs with cell-cluster bootstrap intervals, pooled over the grid.

In [ ]:
ct = pr.paired_table(grid_df, REF, pr.DEFAULT_METRICS, n_boot=N_BOOT) if REF in set(grid_df["arm"]) else None
ct if ct is not None else print("no directed-arm runs yet")

## Wall-clock time per run

Median seconds per run and arm. Times depend on the GPU and on what else ran, so they are reported as medians with the range only.

In [ ]:
grid_df.groupby("arm")["wall_s"].describe()[["count", "min", "50%", "max"]].round(0)